In [ ]:
# v2.0 imports
from systemflow.hep.utils import hep_graph_from_spreadsheet, hep_with_updated_parameters, dataframes_from_spreadsheet
from systemflow.hep.metrics import Productivity
from systemflow.classifier import L1TClassifier, get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import plotly.graph_objects as go

In [ ]:
# Load spreadsheet data
run5_det, run5_proc, run5_globals = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")

In [ ]:
run5_det

In [ ]:
# Fit wall time scaling polynomial
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])
k_gpu, cv_gpu = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time GPU"])

In [ ]:
# Define complexity functions
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}
funcs_gpu = {"Global": lambda x: fit_poly(x, *k_gpu), "Intermediate": lambda x: x / 2.0e6}

In [ ]:
# Build base graph definitions
import warnings
warnings.filterwarnings('ignore')

ex_run5 = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs)
ex_run5_gpu = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs_gpu)

# GPU + L1T tracking variant
l1t_classifier = L1TClassifier(skill_boost=0.40)
ex_run5_l1t_gpu = hep_with_updated_parameters(ex_run5_gpu, {
    "Intermediate": {"classifier (obj)": l1t_classifier}
})

In [ ]:
def extract_results(graph):
    mv = graph.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return power, confusion, graph

In [ ]:
def vary_system(base_def, it_reduction_factor):
    """Vary Inner Tracker data by a multiplicative factor."""
    orig_data = base_def.get_node("Inner Tracker").parameters["sample data (B)"]
    variant = hep_with_updated_parameters(base_def, {
        "Inner Tracker": {"sample data (B)": orig_data * it_reduction_factor}
    })
    result = variant()
    mv = result.metric_values
    return mv["total power (W)"], mv["pipeline contingency (2x2)"], result

In [ ]:
baseline = vary_system(ex_run5, 0.0)

In [ ]:
baseline[0] / 1e6 / density_scale_model(2032)

In [ ]:
# Vary Inner Tracker data reduction from 1.0 to 0.40
it_reductions = np.linspace(1.0, 0.40, 101)

In [ ]:
res_r5 = [vary_system(ex_run5, r) for r in it_reductions]

In [ ]:
res_r5_gpu = [vary_system(ex_run5_gpu, r) for r in it_reductions]

In [ ]:
res_r5_l1t_gpu = [vary_system(ex_run5_l1t_gpu, r) for r in it_reductions]

In [ ]:
def extract_metrics(results):
    all_confusion = np.array([r[1] for r in results])
    all_power = np.array([r[0] / density_scale_model(2032) for r in results])
    all_recall = np.array([recall(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_precision = np.array([precision(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_f1 = np.array([f1_score(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    productivity = np.array([np.sum(get_passed(all_confusion[i,:,:])) for i in range(all_confusion.shape[0])])

    return {
        "confusion": all_confusion,
        "power": all_power,
        "recall": all_recall,
        "precision": all_precision,
        "f1 score": all_f1,
        "productivity": all_recall * productivity
    }

In [ ]:
run5_metrics = extract_metrics(res_r5)
run5_metrics_gpu = extract_metrics(res_r5_gpu)
run5_metrics_l1t_gpu = extract_metrics(res_r5_l1t_gpu)

In [ ]:
run5_metrics_l1t_gpu["f1 score"]

In [ ]:
output_rate = 7.5e3

In [ ]:
fig = go.Figure(data = 
                go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics["f1 score"] * output_rate / run5_metrics["power"] * 1000,
                           name = "Baseline"))

fig.add_trace(go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics_gpu["f1 score"] * output_rate / run5_metrics_gpu["power"] * 1000,
                           name = "+GPU"))


fig.add_trace(go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics_l1t_gpu["f1 score"] * output_rate / run5_metrics_l1t_gpu["power"] * 1000,
                           name = "+L1 Tracks, GPU"))

fig.update_layout(width =800, height = 600,
                  title = "Productivity of DAQ Systems by Inner Tracker Data Reduction",
                  xaxis_title = "Inner Tracker Data Reduction (%)",
                  yaxis_title = "Productivity (Relevant Samples/kJ)")
fig.add_annotation(x = -0.1, 
                   y = -0.15, 
                   showarrow=False,
                   text = "Pileup = 200<br>L1T Rejection = 53:1", 
                   xref="paper", 
                   yref="paper",
                   font = dict(size = 14))
fig.show()

In [ ]:
fig = go.Figure(data = 
                go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics["power"] ,
                           name = "Baseline"))

fig.add_trace(go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics_gpu["power"],
                           name = "+GPU"))


fig.add_trace(go.Scatter(x = (1 - it_reductions)* 100,
                           y = run5_metrics_l1t_gpu["power"],
                           name = "+L1 Tracks, GPU"))

fig.update_layout(width =800, height = 600,
                  title = "Total DAQ Power by Inner Tracker Data Reduction",
                  xaxis_title = "Inner Tracker Data Reduction (%)",
                  yaxis_title = "DAQ Power (W)")
fig.add_annotation(x = -0.1, 
                   y = -0.15, 
                   showarrow=False,
                   text = "Pileup = 200<br>L1T Rejection = 53:1", 
                   xref="paper", 
                   yref="paper",
                   font = dict(size = 14))
fig.show()